# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is a [Croissant schema](https://mlcommons.org/croissant/) JSON-LD, available via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's enumerate all record sets and their columns. You can access record set and field IDs for further referencing.

**Note:** All Croissant entities are referenced by their `@id` field.

In [ ]:
# List all available record sets, their @id, name, description, and columns
print("Available record sets:")
record_sets = list(dataset.record_sets)
if not record_sets:
    # If no record sets are found, print this info
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"- RecordSet @id: {rs.id}")
        print(f"  Name: {getattr(rs, 'name', 'n/a')}")
        print(f"  Description: {getattr(rs, 'description', 'n/a')}")
        print("  Columns:")
        for column in rs.columns:
            print(f"    - Column @id: {column.id} | Name: {getattr(column, 'name', 'n/a')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and column `@id`s from the overview.
Below, we will collect all available record set IDs, extract records for each, and load them into pandas DataFrames referenced by their record set `@id`.

In [ ]:
# Get all record set @ids
record_sets = list(dataset.record_sets)
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"Columns: {df.columns.tolist()}")
    dataframes[record_set_id] = df
    # Display sample records
    display(df.head())

# If there is at least one record set, pick the first for further analysis
default_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

We'll demonstrate basic EDA on the first available record set (by `@id`).

In [ ]:
import numpy as np

if default_record_set_id is not None:
    df = dataframes[default_record_set_id]
    # Identify numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric columns in record set {default_record_set_id}: {numeric_cols}")

    # If there are numeric fields, pick the first one for analysis
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")

        # Set a threshold as the first quantile
        threshold = df[numeric_field].quantile(0.25)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (Q1): {len(filtered_df)} records")
        display(filtered_df[[numeric_field]].head())

        # Normalize numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a non-numeric field (if available)
        group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No non-numeric fields available for grouping.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the chosen numeric field and a scatter plot versus another numeric attribute (if exists).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if default_record_set_id is not None and numeric_cols:
    # Histogram of the chosen numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If there is a second numeric field, show scatter plot
    if len(numeric_cols) > 1:
        y_field = numeric_cols[1]
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[numeric_field], y=df[y_field])
        plt.title(f'{numeric_field} vs. {y_field}')
        plt.xlabel(numeric_field)
        plt.ylabel(y_field)
        plt.show()
else:
    print("Insufficient numeric data for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR² dataset on extracellular vesicle protein analysis using `mlcroissant`.
- We loaded the dataset and displayed available record sets and their fields referenced by their `@id`s.
- We extracted data into pandas DataFrames and demonstrated filtering, normalization, and grouping for numeric fields.
- Visualizations provided insights into distribution and relationships for measurable attributes.

You can extend this notebook for advanced analyses such as predictive modeling, advanced data cleaning, or domain-specific statistics!